**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/clark-2026"], check=True)
    os.chdir("/content/encore/entries/clark-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Clark, Abbott & Sompolinsky (2026): the simulations of the generative-process and rank-four networks

Paper. doi:10.1038/s41593-026-02452-4; claims file `claims/clark-2026.yaml`. Dataset.
DANDI:000939 at version 0.240528.1542, the version the paper cites (see `tuning_curves.ipynb`).

These three claims use no recorded data (`data: elsewhere`): each runs a notebook of the
paper's code at commit 245726f, repeated here in numpy (`hdring.py`). The networks are built
from the idealized generative process (`ring/tc.py`, `gen_tuning_curves_idealized`; sigma
1.4211, beta 2.7659, 500 angles, lambda 1e-8) or from random rank-four loadings. Every random
draw comes from the claim's seed; the paper's notebooks use numpy seeds 42 and 123.

| claim | data | tested here |
|---|---|---|
| figs1-s2-spectrum-does-not-fix-symmetry | elsewhere | yes |
| figs11-s12-global-stability-model | elsewhere | yes |
| figs10-robustness-comparable | elsewhere | yes |
| fig8-grid-scores-and-torus | elsewhere | no, blocked (too large) |

In [ ]:
import os, time, resource
import numpy as np
import pandas as pd
import scipy.linalg
import hdring as H
import ring_dmft as D

T0 = time.time()
SCALE, BETA, N_THETA, LAMDA = 1.4211, 2.7659, 500, 1e-8

In [ ]:
# The generative-process network of 2,500 neurons under the seed of figs11-s12, as the check of
# what the simulations below are built from (no recorded data enter this notebook).
sd11 = H.seed("figs11-s12-global-stability-model")
X25, P25 = H.gen_idealized(2500, N_THETA, np.random.default_rng(sd11), SCALE, BETA)
J25 = H.compute_J(X25, P25, LAMDA)
print("X, Phi", X25.shape, "| J", J25.shape, "| diagonal of J zero:", bool(np.allclose(np.diag(J25), 0)))
print(f"input currents: mean {X25.mean():.3f}, sd {X25.std():.3f} (the process has unit variance); "
      f"firing rates range {P25.min():.3f} to {P25.max():.3f} (1 + erf, bounded by 0 and 2)")
print(f"reconstruction error mean((X - Phi J^T)^2) {np.mean((X25 - P25 @ J25.T) ** 2):.3e}")

## figs1-s2-spectrum-does-not-fix-symmetry

> Eigenvalue degeneracy suggests but does not fix the symmetry group, since ensembles matching
> the model's second-order statistics, and hence its spectrum, can differ beyond second order and
> give different symmetry groups and attractors (a single ring for Gaussian loadings, the full
> torus for mixture-of-Gaussians loadings, in a rank-four network).

As `weight_matrix_analyses.ipynb`, cells 16 and 19: N 10,000, J = M M^T / N, loading standard
deviations 10 and 5 for the two doublets; the mixture gives each neuron one doublet with
standard deviation sqrt(2) times as large, so that the second moments are the same. Dynamics x <-
(1 - dt) x + dt J tanh(x), dt 0.1 for 10 tau, 40 initial conditions with evenly spaced angles in
both doublets, permuted independently (amplitude 5 for the Gaussian, 0.45 in kappa units for the
mixture). J tanh(x) is computed as M (M^T tanh(x)) / N, the same product. The nonzero eigenvalues of
J = M M^T / N are those of the 4 by 4 matrix M^T M / N, which is how they are computed.

Measures (the claims file fixes collapse; the rest are standard choices). kappa in units of the
loading standard deviation, as the paper's notebook. A doublet forms a ring when its 40 final radii
vary by less than 10% (coefficient of variation) and their angles leave no gap wider than 2 pi /
10; it collapses when its median final radius is below one tenth of the ring radius of the
dominant doublet.

In [ ]:
N1, STD1 = 10000, np.array([10., 5.])
def gaussian_M(draw):
    return draw((N1, 4)) * np.repeat(STD1, 2)[None, :]
def mixture_M(group, z):
    M = np.zeros((N1, 4)); comp = np.sqrt(2) * STD1
    for d in range(2):
        M[group == d, 2 * d:2 * d + 2] = comp[d] * z[group == d]
    return M
def eig_lowrank(M):
    return np.sort(np.linalg.eigvalsh(M.T @ M / N1))[::-1]
def run_gauss(M, perm, r0=5.0, n_init=40, dt=0.1, T=10):
    even = np.arange(n_init) * 2 * np.pi / n_init
    ang0 = np.stack([even, perm(even)])
    sub = [M[:, 2 * d:2 * d + 2] / np.std(M[:, 2 * d:2 * d + 2], axis=0) for d in range(2)]
    x = sum(r0 * sub[d] @ np.array([np.cos(ang0[d]), np.sin(ang0[d])]) for d in range(2))
    for i in range(1, int(T / dt)):
        x = (1 - dt) * x + dt * (M @ (M.T @ np.tanh(x)) / N1)
    P = np.tanh(x)
    return [(M[:, 2 * d:2 * d + 2].T @ P) / N1 / np.std(M[:, 2 * d:2 * d + 2], axis=0)[:, None] for d in range(2)], ang0
def run_mix(M, perm, r0=0.45, n_init=40, dt=0.1, T=10):
    even = np.arange(n_init) * 2 * np.pi / n_init
    ang0 = np.stack([even, perm(even)])
    sc = np.repeat([np.std(M[:, 2 * d:2 * d + 2]) for d in range(2)], 2)
    kap0 = np.zeros((4, n_init))
    for d in range(2):
        kap0[2 * d] = r0 * np.cos(ang0[d]); kap0[2 * d + 1] = r0 * np.sin(ang0[d])
    x = (M / sc[None, :]) @ kap0
    for i in range(int(T / dt)):
        P = np.tanh(x)
        traj = (M.T @ P) / N1 / sc[:, None]     # kappa before the step, as the paper's loop
        x = (1 - dt) * x + dt * (M @ (M.T @ P) / N1)
    return [traj[2 * d:2 * d + 2] for d in range(2)], ang0
def ring_stats(kap, ang0):
    r = np.linalg.norm(kap, axis=0); a = np.arctan2(kap[1], kap[0]) % (2 * np.pi)
    s = np.sort(a); gap = np.diff(np.r_[s, s[0] + 2 * np.pi]).max()
    drift = np.abs(np.angle(np.exp(1j * (a - ang0))))
    return dict(radius=np.median(r), cv=r.std() / r.mean(), gap=gap, drift_max=drift.max(),
                ring=bool(r.std() / r.mean() < 0.1 and gap < 2 * np.pi / 10))
def judge_s1(eg, em, kg, ag, km, am):
    rel = np.abs(eg - em) / eg
    g = [ring_stats(kg[d], ag[d]) for d in range(2)]; m = [ring_stats(km[d], am[d]) for d in range(2)]
    g_collapse = g[1]["radius"] < 0.1 * g[0]["radius"]
    m_collapse = min(m[0]["radius"], m[1]["radius"]) < 0.1 * max(m[0]["radius"], m[1]["radius"])
    dyn = g[0]["ring"] and g_collapse and m[0]["ring"] and m[1]["ring"] and not m_collapse
    return rel, g, m, g_collapse, m_collapse, dyn
sd1 = H.seed("figs1-s2-spectrum-does-not-fix-symmetry")
t = time.time()
rows = []
for k in range(5):
    s = sd1 + 1000 * k
    rg = np.random.default_rng(s); rm = np.random.default_rng(s + 1)
    Mg = gaussian_M(rg.standard_normal)
    Mm = mixture_M(rm.integers(0, 2, size=N1), rm.standard_normal((N1, 2)))
    eg, em = eig_lowrank(Mg), eig_lowrank(Mm)
    kg, ag = run_gauss(Mg, np.random.default_rng(s + 2).permutation)
    km, am = run_mix(Mm, np.random.default_rng(s + 3).permutation)
    rel, g, m, gc, mc, dyn = judge_s1(eg, em, kg, ag, km, am)
    rows.append(dict(seed=s, eig_gauss=eg.round(2).tolist(), eig_mix=em.round(2).tolist(), max_rel_diff=round(float(rel.max()), 4),
                     within_5pct=bool(rel.max() <= 0.05), dynamics_as_described=dyn))
    print(f"seed {s}: Gaussian doublet 1 radius {g[0]['radius']:.4f} (cv {g[0]['cv']:.4f}, largest angle gap {g[0]['gap']:.3f}), "
          f"doublet 2 radius {g[1]['radius']:.4f} (collapse {gc}); mixture radii {m[0]['radius']:.4f} (cv {m[0]['cv']:.4f}, gap {m[0]['gap']:.3f}) "
          f"and {m[1]['radius']:.4f} (cv {m[1]['cv']:.4f}, gap {m[1]['gap']:.3f}), rings {m[0]['ring']} {m[1]['ring']}; "
          f"mixture angle drift at most {m[0]['drift_max']:.3f} and {m[1]['drift_max']:.3f} rad")
    if k == 0:
        K0 = (kg, km)
print(pd.DataFrame(rows).to_string(index=False))
print("population eigenvalues (the second moments both ensembles share): 100, 100, 25, 25")
# variant: the paper's own realization, numpy's legacy generator with seed 42 as the notebook seeds it
rs = np.random.RandomState(42); Mg42 = rs.randn(N1, 4) * np.repeat(STD1, 2)[None, :]
rs = np.random.RandomState(42); Mm42 = mixture_M(rs.randint(0, 2, size=N1), rs.randn(N1, 2))
eg42, em42 = eig_lowrank(Mg42), eig_lowrank(Mm42)
print("variant, the paper's seed-42 matrices: Gaussian", eg42.round(2), "| mixture", em42.round(2),
      "| largest relative difference", (np.abs(eg42 - em42) / eg42).max().round(4))
# the mean-field ring radii the paper's notebook prints for the mixture (cell 19)
_z = np.linspace(-30., 30., 200001)
def _E_phi_prime(q):
    w = np.exp(-_z ** 2 / (2 * q)) / np.sqrt(2 * np.pi * q)
    return np.trapezoid((1. - np.tanh(_z) ** 2) * w, _z)
def _solve_q(target):
    lo, hi = 0., 1.
    while _E_phi_prime(hi) > target:
        hi *= 2
    for _ in range(200):
        mid = .5 * (lo + hi)
        lo, hi = (mid, hi) if _E_phi_prime(mid) > target else (lo, mid)
    return .5 * (lo + hi)
print("predicted mixture ring radii (paper's notebook prints 0.5642 0.5636 for its own realization):",
      np.round([np.sqrt(_solve_q(1. / sd ** 2)) / (np.sqrt(2) * sd ** 2) for sd in STD1], 4))
print(f"time {time.time()-t:.0f} s")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 4, figsize=(10, 2.5))
for a_, kap, lab in zip(ax, [K0[0][0], K0[0][1], K0[1][0], K0[1][1]], ["Gaussian, doublet 1", "Gaussian, doublet 2", "mixture, doublet 1", "mixture, doublet 2"]):
    a_.scatter(kap[0], kap[1], s=4); a_.set_xlim(-1, 1); a_.set_ylim(-1, 1); a_.set_aspect(1); a_.set_title(lab, fontsize=8)
plt.tight_layout(); plt.show()
print("final kappa range:", min(float(np.abs(k).max()) for k in K0[0] + K0[1]).__round__(3), "to", max(float(np.abs(k).max()) for k in K0[0] + K0[1]).__round__(3), "(axes -1 to 1)")

**Result: partially reproduced.** The dynamics differ as the legends describe under all five seeds:
with Gaussian loadings the final states form a ring in the dominant doublet (radius 0.80, radii
varying by at most 0.6%, largest angle gap 0.32 to 0.45 rad) and collapse in the subdominant one
(radius 0.012 to 0.024, below one tenth of 0.80); with mixture loadings both doublets form rings
(radii 0.56 to 0.57, the mean-field prediction 0.5642 and 0.5636). The eigenvalues of the two
matrices agree within 5.2% under the claim's seed, just outside the 5% asked (5.18%; 2.4% to 5.2%
over the five seeds, within 5% under two), so that part fails narrowly. The two ensembles share
their second moments (population eigenvalues 100, 100, 25 and 25), and each finite matrix
deviates from them by sampling; the paper's own seed-42 matrices agree within 3.3%. The criterion's 5% applies to two finite
matrices, whose eigenvalues scatter around the shared population values by sampling, so this part
measures sampling noise more than a difference between the ensembles; whether the criterion should
test the population (large-N) spectrum is asked of a reader. The criterion has no rule for a
partial result; partially reproduced is the protocol's reading of a result that holds in part.

## figs11-s12-global-stability-model

> In the generative-process network of 2,500 neurons, initial conditions far from the manifold
> converge to it with no spurious attractors, and with growing N the dynamics approach the
> mean-field behaviour, reliably converging to a ring by N = 1,000.

As `spurious_fixed_points.ipynb`: N 2,500, 500 angles, lambda 1e-8, nonlinearity 1 + erf, no
uniform inhibition, tau 50 ms, Euler step 5 ms, 30 s; 34 initial states i.i.d. Gaussian with
standard deviation 1, 10, 50, 100 and 1,000 (Supplementary Fig. 11a); and for N 50, 100, 250,
500 and 1,000, the 34 regularly spaced manifold points plus noise of standard deviation 0.1
sqrt(N) (Supplementary Fig. 12). The distance to the manifold is the distance of the firing rates to
the nearest of the 500 target points, over sqrt(N), as in `figs11b-data-network-global-stability`.

The criterion judges "as in figs11b", whose threshold is the largest distance at 20 s of the
fig2ef protocol. For this network that protocol is run on the network itself (assumption): the 100
regularly spaced manifold points plus noise of standard deviation 3, with the same step (5 ms),
and the largest distance at 20 s is the threshold. For Supplementary Fig. 12 the threshold is that
of the 2,500-neuron network, a fixed normalized distance (assumption); the threshold of each N's own
reference run is shown as a variant. On the manifold means at least 95% of final states within the
threshold, as in figs11b.

In [ ]:
STDS = (1, 10, 50, 100, 1000)
def model_net(n, rng):
    X, P = H.gen_idealized(n, N_THETA, rng, SCALE, BETA)
    return X, P, H.compute_J(X, P, LAMDA)
def dist_to(P32, Psq, x):
    return H.nearest_target(P32, H.nonlin_idealized(x.astype(np.float32), BETA), Psq)
def reference_run(X, P32, Psq, J, rng):
    # the fig2ef protocol on this network: 100 regularly spaced manifold points plus noise sd 3
    n = X.shape[1]
    x0 = X[::N_THETA // 100].T + rng.standard_normal((n, 100)) * 3.0
    rec = {}
    def on_save(tt, x):
        if abs(tt - 20.0) < 1e-6 or tt == 0.0:
            rec[round(tt)] = dist_to(P32, Psq, x)[1]
    xf = H.simulate(J, x0, T=30.0, dt=0.005, c=0.0, phi=lambda v: H.nonlin_idealized(v, BETA), save_every=0.1, on_save=on_save)
    rec["final"] = dist_to(P32, Psq, xf)[1]
    return rec
def s11a(X, P, J, sd):
    P32 = P.astype(np.float32); Psq = (P32 ** 2).sum(1)
    ref = reference_run(X, P32, Psq, J, np.random.default_rng(sd + 1))
    rng = np.random.default_rng(sd + 2)
    x0 = np.concatenate([rng.standard_normal((X.shape[1], 34)) * sv for sv in STDS], axis=1)
    xf = H.simulate(J, x0, T=30.0, dt=0.005, c=0.0, phi=lambda v: H.nonlin_idealized(v, BETA))
    idx, d = dist_to(P32, Psq, xf); _, d0 = dist_to(P32, Psq, x0)
    out = {sv: dict(init_med=np.median(d0[34 * q:34 * q + 34]), final_med=np.median(d[34 * q:34 * q + 34]), final_max=d[34 * q:34 * q + 34].max(),
                    finite=bool(np.isfinite(xf[:, 34 * q:34 * q + 34]).all()), idx=idx[34 * q:34 * q + 34], d=d[34 * q:34 * q + 34]) for q, sv in enumerate(STDS)}
    return ref, out
t = time.time()
res11 = {}
for k in range(5):
    s = sd11 + 1000 * k
    X, P, J = (X25, P25, J25) if k == 0 else model_net(2500, np.random.default_rng(s))
    ref, out = s11a(X, P, J, s)
    thr = ref[20].max()
    res11[s] = (ref, out, thr)
    print(f"seed {s}: reference (fig2ef protocol) distance at 0 s median {np.median(ref[0]):.3f}, at 20 s median {np.median(ref[20]):.5f}, largest {thr:.5f}, at the end median {np.median(ref['final']):.5f}")
    for sv, o in out.items():
        print(f"   sd {sv}: finite {o['finite']}, distance at the start median {o['init_med']:.3f}, final median {o['final_med']:.5f}, largest {o['final_max']:.5f}, "
              f"within the threshold {np.mean(o['d'] <= thr):.2f}, within 2x {np.mean(o['d'] <= 2 * thr):.2f}; final angles as states {H.count_states(o['idx'], N_THETA, N_THETA / 50)}")
    print(f"   all standard deviations with at least 95% within: {all(np.mean(o['d'] <= thr) >= 0.95 for o in out.values())}; "
          f"strict reading of 'every final state', all within at every standard deviation: {all(bool((o['d'] <= thr).all()) for o in out.values())}", flush=True)
THR25 = res11[sd11][2]
print(f"time {time.time()-t:.0f} s")

In [ ]:
t = time.time()
rows = []
for k in range(5):
    s = sd11 + 1000 * k
    thr = res11[s][2]
    for n in (50, 100, 250, 500, 1000):
        rng = np.random.default_rng(s + 3 + n)
        Xn, Pn, Jn = model_net(n, rng)
        Pn32 = Pn.astype(np.float32); Pnsq = (Pn32 ** 2).sum(1)
        th_idx = (np.arange(34) * N_THETA / 34).round().astype(int)
        x0 = Xn[th_idx].T + rng.standard_normal((n, 34)) * 0.1 * np.sqrt(n)
        xf = H.simulate(Jn, x0, T=30.0, dt=0.005, c=0.0, phi=lambda v: H.nonlin_idealized(v, BETA))
        idx, d = dist_to(Pn32, Pnsq, xf); _, d0 = dist_to(Pn32, Pnsq, x0)
        own = reference_run(Xn, Pn32, Pnsq, Jn, np.random.default_rng(s + 4 + n))[20].max() if k == 0 else np.nan
        cs = H.count_states(idx, N_THETA, N_THETA / 50)
        rows.append(dict(seed=s, N=n, init_median=round(float(np.median(d0)), 3), final_median=float(np.median(d)), final_max=float(d.max()),
                         within_N2500_threshold=round(float(np.mean(d <= thr)), 2), own_threshold=own,
                         within_own=round(float(np.mean(d <= own)), 2) if k == 0 else np.nan, states_bins=cs["bins"], states_single=cs["single"]))
tab12 = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.5f}".format):
    print(tab12.to_string(index=False))
n1000 = tab12[tab12.N == 1000]
print("N 1,000, at least 95% within the 2,500-neuron threshold, by seed:", dict(zip(n1000.seed, n1000.within_N2500_threshold >= 0.95)))
print("N 1,000, strict reading (all 34 within), by seed:", dict(zip(n1000.seed, n1000.within_N2500_threshold >= 1.0)))
print(f"time {time.time()-t:.0f} s")

**Result: reproduced (resting on an assumption).** In the 2,500-neuron network, from every initial
spread (standard deviation 1 to 1,000, median starting distance 1.2 to 1.3) at least 97% of final
states lie within the threshold of the network's own fig2ef-protocol run (largest distance at 20
s 0.0112), none diverges, and all lie within twice it. At N 1,000 all 34 perturbed initial
conditions end within that threshold (largest final distance 0.0106), and the final angles keep 28
of 50 bins. Smaller networks converge less reliably: at N 50 only 6% to 21% of states end within
the threshold over the five seeds. Under four more seeds one (the fourth) falls below 95% at
standard deviation 50 (0.94) and at N 1,000 (0.94); the others hold. Read strictly, "every final state" and "all
converge" fail under the claim's seed at standard deviation 10 (0.97 within: one state at 0.01146
against the threshold 0.01121); the 95% rule is taken from figs11b, which the criterion names as the
way to judge, and it is part of this assumption.

## figs10-robustness-comparable

> Under added weight noise the disordered network and a circulant network with proportional
> eigenvalues fragment into discrete states alike, so the two are comparably robust.

As `weight_matrix_analyses.ipynb`, cell 10: the generative-process network (N 2,500, 500 angles,
lambda 1e-8) and the circulant network built from the mean-field interaction profile
(`DMFTModel(N_theta=2500).J - 1`, scaled so that its diagonal is 10 / N); weight noise of standard
deviation f times the matrix's standard deviation, f 0, 0.2, 0.4, 0.6, 0.8 and 1.0; 34 initial
conditions (regularly spaced manifold points for the disordered network, shifted Gaussian bumps
for the circulant one) with noise of 0.025 times each state's standard deviation; 120 s, step 5
ms, no uniform inhibition. The encoded angle is the argmax of the overlap m (disordered) or of
the input currents (circulant), as the notebook's `theta_max`.

Final angles are merged into states as in `fig2gi-circulant-noise-collapses` (within 2 pi / 50).
The criterion does not say how many states make a continuum among 34 initial conditions spaced 2 pi
/ 34 apart; a network keeps a continuum here when at least 31 (90%) remain distinct under all three
merging readings (assumption).

In [ ]:
NW, NI = 2500, 34
F_W = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
row = D.DMFTModel(N_theta=NW, scale=SCALE, g=BETA).J - 1
J_circ = scipy.linalg.circulant(row); J_circ *= (1.0 / J_circ[0, 0]) * (10.0 / NW)
nn_ = np.arange(NW); nn_ = np.minimum(nn_, NW - nn_)
pat = 4 * np.exp(-(nn_ / (0.15 * NW)) ** 2) - 2
x_circ = np.array([np.roll(pat, r) for r in (np.arange(NI) * NW / NI).round().astype(int)]).T
def setup10(s):
    X, P = H.gen_idealized(NW, N_THETA, np.random.default_rng(s), SCALE, BETA)
    J = H.compute_J(X, P, LAMDA)
    x_dis = X[(np.arange(NI) * N_THETA / NI).round().astype(int)].T
    r2 = np.random.default_rng(s + 1)
    noisy = lambda x: x + r2.standard_normal(x.shape) * 0.025 * x.std(0, keepdims=True)
    return X, P, J, noisy(x_dis), noisy(x_circ)
def angle_of(x, lab, P):
    if lab == "disordered":
        return (P @ H.nonlin_idealized(x.astype(np.float64), BETA) / NW).argmax(0), N_THETA
    return x.argmax(0), NW
def run10(Jb, x0, f, lab, P, rng):
    Jn = Jb + rng.standard_normal(Jb.shape) * f * Jb.std()
    rec = {}
    def on_save(tt, x):
        if round(tt, 3) % 30 == 0:
            rec[round(tt)] = angle_of(x, lab, P)[0]
    xf = H.simulate(Jn, x0, T=120.0, dt=0.005, c=0.0, phi=lambda v: H.nonlin_idealized(v, BETA), save_every=30.0, on_save=on_save)
    a, n = angle_of(xf, lab, P)
    rec["final"] = a
    return rec, n
def summarize10(rec, n):
    cs = H.count_states(rec["final"], n, n / 50)
    drift = np.abs(((rec["final"] - rec[0]) / n * 2 * np.pi + np.pi) % (2 * np.pi) - np.pi)
    return cs, float(np.median(drift)), float(drift.max())
sd10 = H.seed("figs10-robustness-comparable")
t = time.time()
X10, P10, J10, x0d, x0c = setup10(sd10)
rw = np.random.default_rng(sd10 + 2)
rows = []
for f in F_W:
    for lab, Jb, x0 in (("disordered", J10, x0d), ("circulant", J_circ, x0c)):
        rec, n = run10(Jb, x0, f, lab, P10, rw)
        cs, dmed, dmax = summarize10(rec, n)
        st_t = {tt: H.count_states(rec[tt], n, n / 50)["bins"] for tt in (30, 60, 90)}
        rows.append(dict(f_w=f, network=lab, single=cs["single"], greedy=cs["greedy"], bins=cs["bins"], bins_at_30_60_90s=list(st_t.values()),
                         median_drift_rad=round(dmed, 3), max_drift_rad=round(dmax, 3), continuum=min(cs.values()) >= 31))
    print(f"f_w {f} done at {time.time()-t:.0f} s", flush=True)
tab10 = pd.DataFrame(rows)
print(tab10.to_string(index=False))
c0 = tab10[tab10.f_w == 0.0].continuum.all(); c4 = (~tab10[tab10.f_w == 0.4].continuum).all()
print(f"both keep a continuum at f_w 0: {c0}; neither does at 0.4: {c4}")
# seed check at f_w 0.4, the level the criterion reads for fragmentation (the network, the initial noise and the weight noise)
for k in range(1, 5):
    s = sd10 + 1000 * k
    _, Pk, Jk, xdk, xck = setup10(s)
    rwk = np.random.default_rng(s + 2)
    line = []
    for lab, Jb, x0 in (("disordered", Jk, xdk), ("circulant", J_circ, xck)):
        rec, n = run10(Jb, x0, 0.4, lab, Pk, rwk)
        cs, dmed, _ = summarize10(rec, n)
        line.append(f"{lab} states {cs}, median drift {dmed:.3f} rad")
    print(f"seed {s}, f_w 0.4: " + "; ".join(line), flush=True)
print(f"time {time.time()-t:.0f} s")

**Result: reproduced (resting on an assumption).** Without weight noise both networks keep all 34
states (34 under every merging reading, median drift 0.013 rad for the disordered network and
0.025 for the circulant one). At noise 0.4 neither does: the disordered network ends in 8 states
and the circulant in 9 (4 to 8 and 9 to 12 over five seeds), so the criterion is met. The two do
not fragment alike in detail. At noise 0.4 the disordered network ends in fewer states than the
circulant under every seed (8, 7, 4, 8 and 5 against 9, 12, 11, 12 and 9 under bins) with a larger
median drift (0.23 to 0.48 rad against 0.15 to 0.22). Under the claim's seed it keeps fewer states
at every level from 0.4 on (8, 8, 6 and 4 against 9, 12, 9 and 8) and more at 0.2 (15 against
11), and its median drift is larger at every level but 0.8 (0.29 to 0.47 rad from 0.2 on, against
0.16 to 0.30). Both already fragment at noise 0.2, which the criterion does not test. The
statement's "comparably robust" holds in that both lose the continuum at the same noise levels;
the disordered network fragments somewhat further.

## fig8-grid-scores-and-torus

Blocked (too large). The network of 25,000 neurons needs a 25,000 by 25,000 weight matrix (5 GB
in float64, 2.5 GB in float32), fitted by regression on 25,000 by 25,000 kernels and simulated,
and the grid-score sweep 1,000 maps for each eta. It was not started: the five notebooks of this
entry already take about two hours per run (the run before this revision, 1,147, 2,256, 613, 157
and 3,044 s), and this notebook alone about 51 minutes of the 60 allowed.

## Summary

| claim | outcome |
|---|---|
| figs1-s2-spectrum-does-not-fix-symmetry | partially reproduced (seed sensitive) |
| figs11-s12-global-stability-model | reproduced (assumed, seed sensitive) |
| figs10-robustness-comparable | reproduced (assumed) |
| fig8-grid-scores-and-torus | blocked (too large) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Differences from the paper's method. The paper's torch simulations are repeated in numpy
float32; every random draw comes from the claim's seed, not numpy seeds 42 and 123; the rank-four
products J tanh(x) are taken through the loadings, and the eigenvalues through the 4 by 4 Gram
matrix, which are the same quantities. The thresholds for convergence (figs11-s12) and for a
continuum (figs10) are this attempt's assumptions, stated in each section.

In [ ]:
print(f"total time {time.time()-T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")